# Data Pipeline Kualitas Udara

# Notebook 01: Data Ingestion & Basic Cleaning

## Deskripsi
Tahap awal dalam proyek ini adalah mengambil data mentah (*raw data*) dan melakukan pembersihan dasar agar data layak untuk dianalisis lebih lanjut.

## Alur Kerja (Workflow)
1. **Load Raw Data**: Memanggil data asli dari folder `data/raw/`.
2. **Data Definition**: Memahami arti dari setiap kolom yang ada.
3. **Basic Cleaning**:
   - Menghapus data duplikat.
   - Merapikan format nama kolom (standardization).
   - Penyesuaian tipe data (terutama kolom tanggal/datetime).
4. **Data Export**: Menyimpan data hasil pembersihan awal ke folder `data/interim/`.

## Kesimpulan
Data sudah bebas dari duplikat dan memiliki format yang konsisten. Data ini siap untuk masuk ke tahap Eksplorasi (EDA).

## Cell 1: Inisialisasi & Config

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import joblib

# 1. Atur akses ke folder src
parent_dir = os.path.abspath("..")
src_path = os.path.join(parent_dir, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

import utils

# 2. Muat konfigurasi
config = utils.load_config("../config/config.yaml")

print("Konfigurasi dan modul utilitas berhasil dimuat.")

Konfigurasi dan modul utilitas berhasil dimuat.


## Cell 2: Kumpulan Fungsi (Dapur)

In [2]:
def gabung_dan_bersihkan_tipe(folder_path):
    """Menggabungkan CSV dan memastikan kolom polutan jadi angka."""
    daftar_file = sorted([f for f in os.listdir(folder_path) if f.endswith('.csv')])
    list_df = []
    kolom_angka = ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2']
    
    for nama_file in daftar_file:
        df_temp = pd.read_csv(os.path.join(folder_path, nama_file))
        if 'categori' in df_temp.columns:
            df_temp = df_temp.rename(columns={'categori': 'category'})
        for col in kolom_angka:
            if col in df_temp.columns:
                df_temp[col] = pd.to_numeric(df_temp[col], errors='coerce')
        list_df.append(df_temp)
    return pd.concat(list_df, ignore_index=True)

def validasi_logika_data(df_input):
    """Membersihkan data dari duplikat dan nilai negatif."""
    df_clean = df_input.copy()
    print("--- Memulai Pengecekan Kualitas Data ---")
    
    jml_duplikat = df_clean.duplicated().sum()
    if jml_duplikat > 0:
        df_clean = df_clean.drop_duplicates()
        print(f"{jml_duplikat} data duplikat sudah dihapus.")
    
    kolom_polutan = ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2']
    for col in kolom_polutan:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
        negatif = (df_clean[col] < 0).sum()
        if negatif > 0:
            print(f"Kolom {col}: Ada {negatif} nilai negatif! Kita ubah jadi NaN.")
            df_clean.loc[df_clean[col] < 0, col] = np.nan
            
    print("Pengecekan selesai.")
    return df_clean

def check_missing_values(df):
    """Ringkasan missing value."""
    print("--- Ringkasan Missing Value ---")
    null_counts = df.isnull().sum()
    null_percent = (df.isnull().mean() * 100).round(2)
    missing_df = pd.DataFrame({'Jumlah Missing': null_counts, 'Persentase (%)': null_percent})
    print(missing_df.sort_values(by='Jumlah Missing', ascending=False))
    return missing_df

def serialize_data(df, folder_path, file_name):
    """Menyimpan data ke format pickle."""
    os.makedirs(folder_path, exist_ok=True)
    save_path = os.path.join(folder_path, file_name)
    joblib.dump(df, save_path)
    print(f"Hore! Data berhasil diserialisasi ke: {save_path}")
    return save_path

def verify_serialization(path_file, df_original):
    """Verifikasi file .pkl yang sudah disimpan."""
    print("--- Memulai Verifikasi Data Interim ---")
    if os.path.exists(path_file):
        df_verify = joblib.load(path_file)
        if df_verify.shape == df_original.shape:
            print("VERIFIKASI SUKSES: Data identik.")
        else:
            print("PERINGATAN: Dimensi berbeda!")
    else:
        print("ERROR: File tidak ditemukan!")

## Cell 3: Eksekusi Alur Data (Main Program)

In [3]:
# --- EKSEKUSI DENGAN LAPORAN LENGKAP ---

# 1. Gabungkan data mentah
# Mengambil path dari config -> paths -> raw_dataset_dir
path_raw = "../" + config['paths']['raw_dataset_dir'] 
df_raw = gabung_dan_bersihkan_tipe(path_raw)
print(f"Dimensi data AWAL: {df_raw.shape}")

# 2. Validasi (Hapus duplikat & nilai negatif)
df_bersih = validasi_logika_data(df_raw)
print(f"Dimensi data AKHIR: {df_bersih.shape}")

# 3. Cek Missing Value
summary = check_missing_values(df_bersih)

# 4. Simpan file
# PERBAIKAN DI SINI: Tambahkan ['paths'] sebelum nama kuncinya
path_interim = "../" + config['paths']['interim_dataset_dir']
nama_file = config['paths']['interim_filename'] # <--- Ini kuncinya!

file_terpilih = serialize_data(df_bersih, path_interim, nama_file)

# 5. Verifikasi
verify_serialization(file_terpilih, df_bersih)

Dimensi data AWAL: (1830, 11)
--- Memulai Pengecekan Kualitas Data ---
155 data duplikat sudah dihapus.
Pengecekan selesai.
Dimensi data AKHIR: (1675, 11)
--- Ringkasan Missing Value ---
          Jumlah Missing  Persentase (%)
so2                  113            6.75
pm25                  97            5.79
pm10                  68            4.06
o3                    64            3.82
no2                   35            2.09
co                    32            1.91
critical              16            0.96
category               1            0.06
tanggal                0            0.00
stasiun                0            0.00
max                    0            0.00
Hore! Data berhasil diserialisasi ke: ../data/interim/data_gabungan.pkl
--- Memulai Verifikasi Data Interim ---
VERIFIKASI SUKSES: Data identik.
